# 06.3 — Selecting, Filtering, and Slicing

## Learning Objectives

By the end of this notebook you should be able to:

- Select specific rows and columns with `.loc[]` (label-based)
- Select specific rows and columns with `.iloc[]` (position-based)
- Filter rows with boolean masks — single and combined conditions
- Use `.isin()` and `~` (NOT) for common filtering patterns
- Write readable filters with `.query()`
- Explain the copy-vs-view distinction and avoid `SettingWithCopyWarning`

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
print(df.shape)
df.head()

## 1. Column Selection (Review)

- `df["col"]` — one column → `Series`
- `df[["col1", "col2"]]` — multiple columns → `DataFrame`

In [ ]:
df["sex"].head()

In [ ]:
df[["name", "sex", "age", "survived"]].head()

## 2. Row Selection with `.loc[]`

`.loc[]` is the **label-based** selector. With the default integer index, labels *are* the row numbers, but `.loc[]` always means "give me the row whose index label is this value."

```python
df.loc[row_selector, column_selector]
```

Either selector can be a single label, a list, a slice, or a boolean Series. Omit the column selector to get all columns.

In [ ]:
# Single row, all columns
df.loc[0]

In [ ]:
# Single row, specific columns
df.loc[0, ["name", "age", "survived"]]

In [ ]:
# Slice of rows (loc end is INCLUSIVE)
df.loc[0:4]

In [ ]:
# Specific rows and specific columns
df.loc[[0, 5, 10], ["name", "pclass", "survived"]]

## 3. Row Selection with `.iloc[]`

`.iloc[]` is the **position-based** selector. It always uses 0-based integer positions, like Python list indexing. The end of a range is **exclusive**.

In [ ]:
# First row
df.iloc[0]

In [ ]:
# Rows 0–4, columns 0–3
df.iloc[0:5, 0:4]

In [ ]:
# Last 3 rows, first 2 columns
df.iloc[-3:, :2]

### `.loc[]` vs `.iloc[]`

| Scenario | Use |
|---|---|
| Index has meaningful labels (names, dates) | `.loc[]` |
| You want rows in a position range | `.iloc[]` |
| Filtering by a condition | `.loc[]` with a boolean mask |
| Position-based loop | `.iloc[]` |

## 4. Boolean Masking

The most common filtering pattern: create a boolean Series with your condition, then pass it to `.loc[]` (or `[]`) to keep matching rows.

In [ ]:
# Passengers who survived
survivors = df.loc[df["survived"] == 1]
print(f"Survivors: {len(survivors)} out of {len(df)}")
survivors.head()

In [ ]:
# First-class passengers
first_class = df.loc[df["pclass"] == 1]
first_class[["name", "pclass", "fare"]].head()

### Combining Conditions

Use `&` (AND) and `|` (OR). **Always wrap each condition in parentheses** — otherwise Python's operator precedence will cause unexpected errors.

In [ ]:
# First-class passengers who survived
first_class_survivors = df.loc[(df["pclass"] == 1) & (df["survived"] == 1)]
print(f"First-class survivors: {len(first_class_survivors)}")
first_class_survivors[["name", "pclass", "survived", "fare"]].head()

In [ ]:
# Passengers from class 1 OR class 2
upper_class = df.loc[(df["pclass"] == 1) | (df["pclass"] == 2)]
print(f"First and second class: {len(upper_class)}")

### `.isin()` — Cleaner Multi-Value Filtering

When you want to check if a value is in a set of options, `.isin()` is cleaner than chaining `|` conditions.

In [ ]:
# Equivalent to (pclass == 1) | (pclass == 2)
upper_class2 = df.loc[df["pclass"].isin([1, 2])]
print(f"First and second class: {len(upper_class2)}")

### `~` for Negation (NOT)

In [ ]:
# Passengers who did NOT survive
did_not_survive = df.loc[~(df["survived"] == 1)]
print(f"Did not survive: {len(did_not_survive)}")

## 5. `.query()` — Readable Filtering

`.query()` accepts a string expression using column names directly. Many people find it more readable for simple conditions.

In [ ]:
# Equivalent to df.loc[(df["pclass"] == 1) & (df["age"] > 40)]
df.query("pclass == 1 and age > 40")[["name", "pclass", "age", "fare"]].head()

In [ ]:
df.query("pclass in [1, 2] and survived == 1")[["name", "pclass"]].head()

`.query()` is concise but less flexible: it does not work well with column names that have spaces, and it cannot handle complex expressions involving new variables. Boolean masks remain the general-purpose tool.

## 6. Copy vs View — Avoiding `SettingWithCopyWarning`

When you filter a DataFrame, pandas may return a **view** (a window into the original data) or a **copy** (independent data). If you try to modify a view, pandas raises `SettingWithCopyWarning` because it is unsure whether you want to modify the original or not.

In [ ]:
# PROBLEM: chained assignment — may not do what you expect
survivors = df[df["survived"] == 1]
survivors["fare"] = 0   # SettingWithCopyWarning — are we modifying a copy or the original?

In [ ]:
# SOLUTION: call .copy() when you intend to work with a separate DataFrame
survivors = df[df["survived"] == 1].copy()
survivors["fare"] = 0   # no warning — clearly modifying an independent copy

# Verify the original is unchanged
print("Original first fare:", df["fare"].iloc[1])

**Rule of thumb:** call `.copy()` immediately after any filter when you plan to *modify* the result. If you only need to *read* from it, `.copy()` is unnecessary.

## Putting It Together

*Which women in third class survived, and what did they pay?*

In [ ]:
third_class_women_survivors = df.loc[
    (df["pclass"] == 3) &
    (df["sex"] == "female") &
    (df["survived"] == 1),
    ["name", "age", "fare"]
].copy()

print(f"Count: {len(third_class_women_survivors)}")
third_class_women_survivors.sort_values("fare", ascending=False).head(10)

## Your Turn

1. Select only passengers who were children (age < 18). How many are there?
2. Find passengers who paid more than £100 AND did not survive. Display `name`, `pclass`, `fare`, `survived`.
3. Find all passengers whose name contains `"Miss."`. How many? *(Hint: `.str.contains()`)*
4. Using `.iloc[]`, select passengers at positions 100 through 109 and display `name`, `age`, `pclass`.
5. Select male passengers in first class. Then select only female passengers who did NOT survive. Which group is larger?

In [ ]:
# Your code here